# 03_build_current_state
Read one already validated run, normalize domains, remove IP literals, deduplicate within feeds, and deduplicate globally.

Normalization removes only a leading `www` or `www<number>` label when the resulting hostname exists in this run. `example.com`, `www.example.com`, and `www2.example.com` become `example.com` only when that parent is present. Without it, sibling subdomains remain separate. Matching uses complete DNS labels.

The output replaces the entire current table. Run serially within a schema; older snapshots are rejected.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import ipaddress
import json

In [0]:
%run ./utils/functions

In [0]:
%run ./utils/validations

### Configure parameters

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("ingestion_ts", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "pb_threat_intelligence")

In [0]:
run_id = dbutils.widgets.get("run_id").strip()
ingestion_ts = datetime.fromisoformat(
    dbutils.widgets.get("ingestion_ts").strip()
)
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

audit_table = f"{catalog}.{schema}.feed_count_audit"
observations_table = f"{catalog}.{schema}.feed_observations"
output_table = f"{catalog}.{schema}.domains_classification_current"

# Configuration checks only; feed validation belongs to the preceding task.
if not run_id:
    raise ValueError("An explicit run_id is required")

### Read and clean observations

In [0]:
# Reject an older snapshot in a schema that already contains a newer state.
# Replay older inputs in a separate schema. Run jobs serially within each schema.
newer = spark.table(output_table).filter(F.col("built_at") > F.lit(ingestion_ts)).limit(1).count()
if newer:
    raise ValueError("Older snapshot: use a separate schema for replay")

# Clear derived counts for this run before rebuilding, avoiding stale retry entries.
from delta.tables import DeltaTable
DeltaTable.forName(spark, audit_table).delete(
    (F.col("run_id") == run_id) & F.col("stage").isin("filtered out", "current domains")
)

# Read only observations belonging to the requested snapshot run.
run_observations = spark.table(observations_table).filter(F.col("run_id") == run_id)

# Standardize case, whitespace, and the optional DNS trailing dot.
cleaned = run_observations.withColumn(
    "domain", F.regexp_replace(F.lower(F.trim(F.col("domain"))), r"\.$", "")
)

# filtering out IPs and wrong formats
domains_only = cleaned.filter(F.col("validation_status") == "accepted domain")

### Count filtered-out records

In [0]:
# Count filtered-out records by run, feed, and rejection reason.
filtered_out_counts = (
    cleaned
    .filter(
        F.col("validation_status").isNull()
        | (F.col("validation_status") != "accepted domain")
    )
    .groupBy("run_id", "feed_name", "validation_status")
    .count()
)

print(f"Filtered out records:")
filtered_out_counts.display()

# Write one audit entry for each run, feed, and validation status.
for row in filtered_out_counts.collect():
    log_record_count(
        audit_table=audit_table,
        run_id=row["run_id"],
        run_date=ingestion_ts.date(),
        feed_name=row["feed_name"],
        stage="filtered out",
        record_count=row["count"],
        status=row["validation_status"],
    )

## Map www variants to an existing parent
Only the leading www label is considered. Other subdomains remain unchanged. Matching stays in Spark.

In [0]:
existing_domains = domains_only.select("domain").distinct()

# find domains with leading www / www<number> label
domain_candidates = existing_domains.select(
    F.col("domain").alias("original_domain"),
    F.regexp_replace(
        F.col("domain"),
        r"^www\d*\.",
        "",
    ).alias("candidate"),
)

# match the candidate only if it already exists in this run.
existing_parents = existing_domains.select(
    F.col("domain").alias("existing_parent")
)

domain_mapping = (
    domain_candidates
    .join(
        existing_parents,
        F.col("candidate") == F.col("existing_parent"),
        how="left",
    )
    .select(
        "original_domain",
        F.coalesce(
            F.col("existing_parent"),
            F.col("original_domain"),
        ).alias("domain"),
    )
)

# map www variants to their parent
normalized_observations = (
    domains_only
    .withColumnRenamed("domain", "original_domain")
    .join(domain_mapping, on="original_domain", how="inner")
).drop("original_domain")

# Collect provenance after mapping, before any duplicate/category removal.
# All source feeds are retained, including feeds with the losing category.
feeds = (normalized_observations.groupBy("domain")
         .agg(F.sort_array(F.collect_set("feed_name")).alias("feeds"))
         .withColumn("feed_count", F.size("feeds")))
normalized_observations = normalized_observations.join(feeds, "domain", "left")

### Deduplicate and rank categories

In [0]:
# filter out duplicated domains falling into the same category
deduplicated = normalized_observations.dropDuplicates(["category", "domain"])
duplicates_count = normalized_observations.count() - deduplicated.count()

# log the number of duplicates
log_record_count(
    audit_table=audit_table,
    run_id=run_id,
    run_date=ingestion_ts.date(),
    feed_name="all feeds",
    stage="filtered out",
    record_count=duplicates_count,
    status="duplicates per category",
)

In [0]:
# rank distinct categories for each domain
# assume that the lowest priority category is the most relevant
# meaning that malware is more relevant than ads_and_trackers

category_order = Window.partitionBy("domain").orderBy(F.col("category_priority").asc())

current_state = (
    deduplicated
    .withColumn(
        "category_priority",
        F.when(F.col("category") == "malware", 1)
        .when(F.col("category") == "ads_and_trackers", 2)
        .otherwise(999),
    )
    .withColumn("rank", F.row_number().over(category_order))
    .filter(F.col("rank") == 1)
).drop("rank", "category_priority")

In [0]:

# store logs in audit table
log_record_count(
    audit_table=audit_table,
    run_id=run_id,
    run_date=ingestion_ts.date(),
    feed_name="all feeds",
    stage="filtered out",
    record_count=deduplicated.count() - current_state.count(),
    status="duplicated with different category",
)

### Publish current state

In [0]:
current_state = current_state.select(
    F.col("domain").cast("string"),
    F.col("category").cast("string"),
    F.col("feeds").cast("array<string>"),
    F.col("feed_count").cast("int"),
    F.col("run_id").cast("string"),
    F.lit(ingestion_ts).cast("timestamp").alias("built_at"),
)

In [0]:
# verifies that the current state df is meeting criteria 
validate_current_state(current_state)
validate_count_balance(audit_table, run_id, current_state.count())

In [0]:
# Publish the full current snapshot in a single Delta overwrite.
# The existing output schema must be compatible with the new schema.
current_state.write.format("delta").mode("overwrite").option("mergeSchema", "false").saveAsTable(output_table)
print({"run_id": run_id, "output_table": output_table, "status": "published", "count": current_state.count()})

### Log result

In [0]:
log_record_count(
    audit_table=audit_table,
    run_id=run_id,
    run_date=ingestion_ts.date(),
    feed_name="all feeds",
    stage="current domains",
    record_count=current_state.count(),
    status="published",
)

## Result
One row per canonical `domain`, with `feeds`, `feed_count`, `category`, `run_id`, and `built_at`.

Example: feed A contains `www.example.com` and feed B contains `example.com` and `www2.example.com`. The final row is `example.com`, with both feeds. This means a feed contributed to the canonical domain through either that domain or one of its www variants; it does not mean both feeds explicitly listed the parent.

This mapping assumes equivalence of the www variants and the listed parent